# 🇮🇩 Prediksi Tingkat Kemiskinan di Indonesia
## UTS Praktikum Kecerdasan Buatan

**Notebook ini dokumentasikan:**
- Exploratory Data Analysis (EDA)
- Preprocessing & Feature Engineering
- Training 5 Model Machine Learning
- Evaluasi dan Perbandingan Performa
- Interpretasi Hasil

**Dataset**: Badan Pusat Statistik Indonesia (BPS)  
**Target**: Persentase Penduduk Miskin (P0) per Kab/Kota  
**Scope**: 514 Kabupaten/Kota, 34 Provinsi

---
## 1️⃣ Setup & Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ML Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, 
    silhouette_score, mean_absolute_percentage_error
)

try:
    import tensorflow as tf
    from tensorflow import keras
    TF_AVAILABLE = True
    print(f"✓ TensorFlow {tf.__version__} imported")
except ImportError:
    TF_AVAILABLE = False
    print("⚠ TensorFlow not available (will use Scikit-learn MLP fallback)")

---
## 2️⃣ Load & Inspect Dataset

In [ ]:
# Data path (adjust if needed)
DATA_PATH = Path('../data/Klasifikasi_Tingkat_Kemiskinan_di_Indonesia.csv')

# Load with proper encoding and decimal separator
df = pd.read_csv(DATA_PATH, sep=';', decimal=',')

# Clean column names
df.columns = [c.strip() for c in df.columns]

# Strip whitespace in string columns
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.strip()

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(3)

In [ ]:
# Column info
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Data types dan missing values
info_df = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes,
    'Missing': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
info_df

---
## 3️⃣ Data Cleaning & Preprocessing

In [ ]:
# Target column
TARGET_COL = 'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'

# Feature columns
FEATURE_COLS = [
    'Rata-rata Lama Sekolah Penduduk 15+ (Tahun)',
    'Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun)',
    'Indeks Pembangunan Manusia',
    'Umur Harapan Hidup (Tahun)',
    'Persentase rumah tangga yang memiliki akses terhadap sanitasi layak',
    'Persentase rumah tangga yang memiliki akses terhadap air minum layak',
    'Tingkat Pengangguran Terbuka',
    'Tingkat Partisipasi Angkatan Kerja',
    'PDRB atas Dasar Harga Konstan menurut Pengeluaran (Rupiah)',
]

print(f"Target: {TARGET_COL}")
print(f"\nFeatures ({len(FEATURE_COLS)}):")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"  {i}. {col}")

In [ ]:
# Convert target & features to numeric (replace comma with dot)
for col in [TARGET_COL] + FEATURE_COLS:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.').str.strip(),
        errors='coerce'
    )

# Remove rows with missing values
df_clean = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).copy()

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")

df = df_clean

In [ ]:
# Extract X and y
X = df[FEATURE_COLS].values.astype(np.float64)
y = df[TARGET_COL].values.astype(np.float64)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny (Target) Statistics:")
print(f"  Mean: {y.mean():.4f}%")
print(f"  Std:  {y.std():.4f}%")
print(f"  Min:  {y.min():.4f}%")
print(f"  Max:  {y.max():.4f}%")

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")
print(f"\nTrain-Test split: 80-20")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled using StandardScaler")
print(f"\nScaled X_train statistics:")
print(f"  Mean: {X_train_scaled.mean(axis=0).mean():.6f}")
print(f"  Std:  {X_train_scaled.std(axis=0).mean():.4f}")

---
## 4️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(y, bins=20, color='#1E3A8A', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Persentase Penduduk Miskin (%)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frekuensi (Kab/Kota)', fontsize=11, fontweight='bold')
axes[0].set_title('Distribusi Target: Tingkat Kemiskinan', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Box plot
axes[1].boxplot(y, vert=True)
axes[1].set_ylabel('Persentase Penduduk Miskin (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Box Plot: Tingkat Kemiskinan', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Target Distribution Summary:")
print(f"  Median: {np.median(y):.2f}%")
print(f"  Q1:     {np.percentile(y, 25):.2f}%")
print(f"  Q3:     {np.percentile(y, 75):.2f}%")
print(f"  IQR:    {np.percentile(y, 75) - np.percentile(y, 25):.2f}%")

In [ ]:
# Correlation analysis
corr_data = pd.DataFrame(
    np.column_stack([X, y]),
    columns=FEATURE_COLS + [TARGET_COL]
)

# Calculate correlations with target
correlations = corr_data.corr()[TARGET_COL].drop(TARGET_COL).sort_values()

print("\n🔗 Feature Correlations with Target:")
print(correlations)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#10B981' if x < 0 else '#EF4444' for x in correlations]
correlations.plot(kind='barh', ax=ax, color=colors, alpha=0.8, edgecolor='black')
ax.set_xlabel('Korelasi Pearson', fontsize=11, fontweight='bold')
ax.set_title('Feature Correlation dengan Tingkat Kemiskinan\n(Negatif = Inverse Relationship)', 
             fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Geographical analysis: Poverty by Province
province_poverty = df.groupby('Provinsi')[TARGET_COL].agg(['mean', 'count']).sort_values('mean', ascending=False)
province_poverty.columns = ['Rata-rata Kemiskinan (%)', 'Jumlah Kab/Kota']

print("\n🗺️  Top 10 Provinsi dengan Kemiskinan Tertinggi:")
print(province_poverty.head(10))

print("\n🗺️  Bottom 5 Provinsi dengan Kemiskinan Terendah:")
print(province_poverty.tail(5))

In [ ]:
# Visualize poverty by province
fig, ax = plt.subplots(figsize=(14, 8))
top_prov = province_poverty.head(15)
colors_prov = plt.cm.Reds(np.linspace(0.4, 0.9, len(top_prov)))
top_prov['Rata-rata Kemiskinan (%)'].plot(kind='barh', ax=ax, color=colors_prov, edgecolor='black')
ax.set_xlabel('Rata-rata Persentase Kemiskinan (%)', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Provinsi dengan Tingkat Kemiskinan Tertinggi', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (features only)
feature_corr = corr_data[FEATURE_COLS + [TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(feature_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Korelasi'}, ax=ax)
ax.set_title('Correlation Matrix: Features vs Target', fontsize=12, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 5️⃣ Model 1: Linear Regression

In [ ]:
print("="*70)
print("MODEL 1: LINEAR REGRESSION")
print("="*70)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

# Metrics
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"\n📊 Linear Regression Performance (Test Set):")
print(f"  MAE (Mean Absolute Error):  {mae_lr:.4f}")
print(f"  RMSE (Root MSE):            {rmse_lr:.4f}")
print(f"  R² Score:                   {r2_lr:.4f}")

# Feature importance (coefficients)
feature_importance_lr = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"\n🔧 Feature Importance (Coefficients):")
print(feature_importance_lr.to_string(index=False))

---
## 6️⃣ Model 2: Artificial Neural Network (ANN)

In [ ]:
print("="*70)
print("MODEL 2: ARTIFICIAL NEURAL NETWORK (ANN) - KERAS/TENSORFLOW")
print("="*70)

if TF_AVAILABLE:
    tf.random.set_seed(42)
    
    ann_model = keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1),
    ])
    
    ann_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    print("\n🏗️  ANN Architecture:")
    ann_model.summary()
    
    print("\n⏳ Training ANN...")
    history = ann_model.fit(
        X_train_scaled, y_train,
        epochs=80,
        batch_size=32,
        validation_split=0.1,
        verbose=0
    )
    
    y_pred_ann = ann_model.predict(X_test_scaled, verbose=0).flatten()
    
    mae_ann = mean_absolute_error(y_test, y_pred_ann)
    rmse_ann = np.sqrt(mean_squared_error(y_test, y_pred_ann))
    r2_ann = r2_score(y_test, y_pred_ann)
    
    print(f"\n📊 ANN Performance (Test Set):")
    print(f"  MAE:    {mae_ann:.4f}")
    print(f"  RMSE:   {rmse_ann:.4f}")
    print(f"  R² Score: {r2_ann:.4f}")
    
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Loss (MSE)', fontsize=11, fontweight='bold')
    axes[0].set_title('ANN: Training History - Loss', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
    axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('MAE', fontsize=11, fontweight='bold')
    axes[1].set_title('ANN: Training History - MAE', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    ann_backend = "TensorFlow/Keras"
else:
    print("\n⚠️  TensorFlow not available. Using Scikit-learn MLP as fallback...")
    ann_model = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        max_iter=300,
        random_state=42,
        early_stopping=True,
        learning_rate_init=0.001
    )
    ann_model.fit(X_train_scaled, y_train)
    y_pred_ann = ann_model.predict(X_test_scaled)
    
    mae_ann = mean_absolute_error(y_test, y_pred_ann)
    rmse_ann = np.sqrt(mean_squared_error(y_test, y_pred_ann))
    r2_ann = r2_score(y_test, y_pred_ann)
    
    print(f"\n📊 ANN Performance (Test Set):")
    print(f"  MAE:    {mae_ann:.4f}")
    print(f"  RMSE:   {rmse_ann:.4f}")
    print(f"  R² Score: {r2_ann:.4f}")
    ann_backend = "Scikit-learn MLPRegressor"

---
## 7️⃣ Model 3: LSTM / RNN

In [ ]:
print("="*70)
print("MODEL 3: LSTM / RNN - RECURRENT NEURAL NETWORK")
print("="*70)

if TF_AVAILABLE:
    tf.random.set_seed(42)
    n_features = X_train_scaled.shape[1]
    
    # Reshape data for LSTM: (samples, timesteps, features)
    # Treat each feature column as a timestep
    X_train_lstm = X_train_scaled.reshape(-1, n_features, 1)
    X_test_lstm = X_test_scaled.reshape(-1, n_features, 1)
    
    lstm_model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.LSTM(64, return_sequences=True),
        keras.layers.LSTM(32),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1),
    ])
    
    lstm_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    print("\n🏗️  LSTM Architecture:")
    lstm_model.summary()
    
    print("\n⏳ Training LSTM...")
    history_lstm = lstm_model.fit(
        X_train_lstm, y_train,
        epochs=80,
        batch_size=32,
        validation_split=0.1,
        verbose=0
    )
    
    y_pred_lstm = lstm_model.predict(X_test_lstm, verbose=0).flatten()
    
    mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
    rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
    mape_lstm = mean_absolute_percentage_error(y_test, y_pred_lstm)
    
    print(f"\n📊 LSTM Performance (Test Set):")
    print(f"  MAE:    {mae_lstm:.4f}")
    print(f"  RMSE:   {rmse_lstm:.4f}")
    print(f"  MAPE:   {mape_lstm*100:.4f}%")
    lstm_backend = "TensorFlow/Keras LSTM"
else:
    print("\n⚠️  TensorFlow not available. Using Scikit-learn MLP as fallback...")
    lstm_model = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=42,
        early_stopping=True
    )
    lstm_model.fit(X_train_scaled, y_train)
    y_pred_lstm = lstm_model.predict(X_test_scaled)
    
    mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
    rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
    mape_lstm = mean_absolute_percentage_error(y_test, y_pred_lstm)
    
    print(f"\n📊 LSTM Performance (Test Set):")
    print(f"  MAE:    {mae_lstm:.4f}")
    print(f"  RMSE:   {rmse_lstm:.4f}")
    print(f"  MAPE:   {mape_lstm*100:.4f}%")
    lstm_backend = "Scikit-learn MLP (fallback)"

---
## 8️⃣ Model 4: K-Means Clustering

In [ ]:
print("="*70)
print("MODEL 4: K-MEANS CLUSTERING (UNSUPERVISED)")
print("="*70)

# Fit K-Means on ALL data (unsupervised)
X_all_scaled = scaler.transform(X)
kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans_model.fit_predict(X_all_scaled)

# Metrics
inertia = kmeans_model.inertia_
silhouette = silhouette_score(X_all_scaled, cluster_labels, sample_size=min(400, len(X)))

print(f"\n📊 K-Means Performance (K=3):")
print(f"  Inertia (Sum of Squared Distances): {inertia:.2f}")
print(f"  Silhouette Score: {silhouette:.4f}")
print(f"  Number of Clusters: 3")

# Analyze cluster characteristics
print(f"\n📈 Cluster Analysis:")
cluster_analysis = pd.DataFrame({
    'Cluster': cluster_labels,
    'Poverty': y
})

for cluster_id in range(3):
    cluster_poverty = y[cluster_labels == cluster_id]
    cluster_size = len(cluster_poverty)
    cluster_mean = cluster_poverty.mean()
    print(f"\n  Cluster {cluster_id}:")
    print(f"    Size: {cluster_size} daerah")
    print(f"    Mean Poverty: {cluster_mean:.2f}%")
    print(f"    Min Poverty:  {cluster_poverty.min():.2f}%")
    print(f"    Max Poverty:  {cluster_poverty.max():.2f}%")
    print(f"    Std Dev:      {cluster_poverty.std():.2f}%")

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter by cluster
scatter = axes[0].scatter(cluster_labels, y, c=cluster_labels, cmap='viridis', s=80, alpha=0.6, edgecolor='black')
axes[0].set_xlabel('Cluster ID', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Persentase Kemiskinan (%)', fontsize=11, fontweight='bold')
axes[0].set_title('K-Means Clustering: Distribution by Cluster', fontsize=12, fontweight='bold')
axes[0].set_xticks([0, 1, 2])
axes[0].grid(alpha=0.3)

# Box plot by cluster
cluster_data = [y[cluster_labels == i] for i in range(3)]
bp = axes[1].boxplot(cluster_data, labels=['Cluster 0', 'Cluster 1', 'Cluster 2'])
axes[1].set_ylabel('Persentase Kemiskinan (%)', fontsize=11, fontweight='bold')
axes[1].set_title('K-Means: Box Plot by Cluster', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9️⃣ Model 5: Manual Backpropagation (Pure NumPy)

In [ ]:
print("="*70)
print("MODEL 5: BACKPROPAGATION - MANUAL NEURAL NETWORK (PURE NUMPY)")
print("="*70)

class ManualBackpropNN:
    """Manual feedforward neural network with backpropagation (NumPy only)"""
    
    def __init__(self, layer_sizes, learning_rate=0.005, epochs=400):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.epochs = epochs
        self.losses = []
        self.weights = []
        self.biases = []
        
        np.random.seed(42)
        for i in range(len(layer_sizes) - 1):
            n_in, n_out = layer_sizes[i], layer_sizes[i + 1]
            self.weights.append(np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in))
            self.biases.append(np.zeros((1, n_out)))
    
    @staticmethod
    def _relu(z):
        return np.maximum(0, z)
    
    @staticmethod
    def _relu_derivative(z):
        return (z > 0).astype(float)
    
    def _forward(self, X):
        self._activations = [X]
        self._pre_activations = []
        current = X
        
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = current @ W + b
            self._pre_activations.append(z)
            current = self._relu(z) if i < len(self.weights) - 1 else z
            self._activations.append(current)
        
        return current
    
    def _backward(self, y):
        m = self._activations[0].shape[0]
        delta = (self._activations[-1] - y.reshape(-1, 1)) * (2.0 / m)
        
        for i in reversed(range(len(self.weights))):
            dW = self._activations[i].T @ delta
            db = delta.sum(axis=0, keepdims=True)
            
            if i > 0:
                delta = delta @ self.weights[i].T * self._relu_derivative(self._pre_activations[i - 1])
            
            self.weights[i] -= self.lr * dW
            self.biases[i] -= self.lr * db
    
    def fit(self, X, y):
        for epoch in range(self.epochs):
            y_hat = self._forward(X).flatten()
            loss = np.mean((y_hat - y) ** 2)
            self.losses.append(loss)
            self._backward(y)
            
            if (epoch + 1) % 100 == 0:
                print(f"    Epoch {epoch+1:3d}/{self.epochs}: Loss = {loss:.6f}")
        return self
    
    def predict(self, X):
        return self._forward(X).flatten()
    
    def convergence_rate(self):
        if len(self.losses) < 2:
            return 0.0
        return (self.losses[0] - self.losses[-1]) / max(self.losses[0], 1e-9) * 100

print("\n⏳ Training Backpropagation Network...")
print(f"   Architecture: {X_train_scaled.shape[1]} → 64 → 32 → 1")
print(f"   Learning Rate: 0.005")
print(f"   Epochs: 400\n")

bp_model = ManualBackpropNN(
    layer_sizes=[X_train_scaled.shape[1], 64, 32, 1],
    learning_rate=0.005,
    epochs=400
)
bp_model.fit(X_train_scaled, y_train)

y_pred_bp = bp_model.predict(X_test_scaled)

mae_bp = mean_absolute_error(y_test, y_pred_bp)
rmse_bp = np.sqrt(mean_squared_error(y_test, y_pred_bp))
conv_rate = bp_model.convergence_rate()

print(f"\n✓ Training Complete!")
print(f"\n📊 Backpropagation Performance (Test Set):")
print(f"  MAE:  {mae_bp:.4f}")
print(f"  RMSE: {rmse_bp:.4f}")
print(f"  Initial Loss: {bp_model.losses[0]:.4f}")
print(f"  Final Loss:   {bp_model.losses[-1]:.4f}")
print(f"  Convergence Rate: {conv_rate:.2f}%")

In [ ]:
# Plot convergence curve
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(bp_model.losses, linewidth=2, color='#D97706', alpha=0.8)
ax.fill_between(range(len(bp_model.losses)), bp_model.losses, alpha=0.2, color='#D97706')
ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax.set_ylabel('MSE Loss', fontsize=11, fontweight='bold')
ax.set_title('Backpropagation: Training Loss Convergence', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Add convergence rate annotation
ax.text(0.5, 0.95, f'Convergence Rate: {conv_rate:.2f}%', 
        transform=ax.transAxes, fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        verticalalignment='top', horizontalalignment='center')

plt.tight_layout()
plt.show()

---
## 🔟 Model Comparison & Results

In [ ]:
# Create comparison table
comparison_data = {
    'Model': ['Linear Regression', 'ANN (Keras)', 'LSTM/RNN', 'K-Means', 'Backpropagation'],
    'Type': ['Supervised Regression', 'Supervised DL', 'Supervised Sequential', 'Unsupervised', 'Supervised DL (Manual)'],
    'MAE': [mae_lr, mae_ann, mae_lstm, np.nan, mae_bp],
    'RMSE': [rmse_lr, rmse_ann, rmse_lstm, np.nan, rmse_bp],
    'R² Score': [r2_lr, r2_ann, np.nan, np.nan, np.nan],
    'MAPE (%)': [np.nan, np.nan, mape_lstm*100, np.nan, np.nan],
    'Silhouette': [np.nan, np.nan, np.nan, silhouette, np.nan],
    'Special Metric': [
        f'Coefficients: {len(FEATURE_COLS)}',
        f'Layers: 4 | Params: {(128+1)*9 + (64+1)*128 + (32+1)*64 + 32}',
        f'LSTM: 2 | Dense: 2',
        f'Inertia: {inertia:.2f}',
        f'Conv Rate: {conv_rate:.1f}%'
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*150)
print("COMPREHENSIVE MODEL COMPARISON TABLE")
print("="*150)
print(comparison_df.to_string(index=False))
print("="*150)

In [ ]:
# Visualize regression models' performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models = ['Linear Regression', 'ANN', 'LSTM', 'Backpropagation']
mae_vals = [mae_lr, mae_ann, mae_lstm, mae_bp]
rmse_vals = [rmse_lr, rmse_ann, rmse_lstm, rmse_bp]
colors_models = ['#1E3A8A', '#7C3AED', '#0891B2', '#D97706']

# MAE comparison
axes[0, 0].bar(models, mae_vals, color=colors_models, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('MAE', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Mean Absolute Error (Lower is Better)', fontsize=11, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(mae_vals):
    axes[0, 0].text(i, v + 0.05, f'{v:.3f}', ha='center', fontweight='bold')
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# RMSE comparison
axes[0, 1].bar(models, rmse_vals, color=colors_models, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('RMSE', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Root Mean Squared Error (Lower is Better)', fontsize=11, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(rmse_vals):
    axes[0, 1].text(i, v + 0.05, f'{v:.3f}', ha='center', fontweight='bold')
plt.setp(axes[0, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

# R² comparison (for available models)
r2_models = ['Linear Regression', 'ANN', 'Backpropagation']
r2_vals = [r2_lr, r2_ann, np.nan]  # bp doesn't have r2
axes[1, 0].bar(r2_models[:2], [r2_lr, r2_ann], color=colors_models[:2], alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('R² Score', fontsize=11, fontweight='bold')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].set_title('R² Score (Higher is Better, Max=1.0)', fontsize=11, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)
for i, (model, v) in enumerate(zip(r2_models[:2], [r2_lr, r2_ann])):
    axes[1, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Summary ranking
axes[1, 1].axis('off')
summary_text = f"""BEST MODELS RANKING (by MAE):

🥇 {models[mae_vals.index(min(mae_vals))]}
   MAE: {min(mae_vals):.4f}

🥈 {sorted(zip(mae_vals, models))[1][1]}
   MAE: {sorted(mae_vals)[1]:.4f}

🥉 {sorted(zip(mae_vals, models))[2][1]}
   MAE: {sorted(mae_vals)[2]:.4f}

K-Means Clustering:
  Silhouette: {silhouette:.4f}
  Inertia: {inertia:.2f}

Backpropagation Convergence:
  Rate: {conv_rate:.2f}%
"""
axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Prediction scatter plots (actual vs predicted)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

all_preds = [
    (y_pred_lr, 'Linear Regression', '#1E3A8A'),
    (y_pred_ann, 'ANN (Keras)', '#7C3AED'),
    (y_pred_lstm, 'LSTM/RNN', '#0891B2'),
    (y_pred_bp, 'Backpropagation', '#D97706'),
]

for ax, (pred, name, color) in zip(axes.flat, all_preds):
    ax.scatter(y_test, pred, alpha=0.6, s=60, color=color, edgecolor='black', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    ax.set_xlabel('Actual Poverty (%)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Predicted Poverty (%)', fontsize=10, fontweight='bold')
    ax.set_title(f'{name}\nMAE: {mean_absolute_error(y_test, pred):.4f}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left')

plt.suptitle('Actual vs Predicted: Test Set Performance', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

---
## 1️⃣1️⃣ Kesimpulan & Interpretasi

In [ ]:
print("\n" + "="*80)
print("KESIMPULAN & INTERPRETASI HASIL")
print("="*80)

print("\n📊 DATASET INSIGHTS:")
print(f"  • Total sampel: {len(df)} Kabupaten/Kota dari {df['Provinsi'].nunique()} Provinsi")
print(f"  • Target range: {y.min():.2f}% - {y.max():.2f}% kemiskinan")
print(f"  • Rata-rata kemiskinan: {y.mean():.2f}% (± {y.std():.2f}%)")
print(f"  • Features: {len(FEATURE_COLS)} indikator sosial-ekonomi")

print("\n🎯 TOP 3 PREDICTIVE FEATURES (by correlation):")
top_features = correlations.abs().nlargest(3)
for i, (feat, corr) in enumerate(top_features.items(), 1):
    direction = "negative" if correlations[feat] < 0 else "positive"
    print(f"  {i}. {feat[:40]:40s} | r = {correlations[feat]:7.4f} ({direction})")

print(f"\n📈 MODEL PERFORMANCE SUMMARY:")
print(f"  Model                 │  MAE    │  RMSE   │  R²    │ Special Metric")
print(f"  ─────────────────────────────────────────────────────────────────────")
print(f"  Linear Regression     │  {mae_lr:.4f} │  {rmse_lr:.4f} │ {r2_lr:.4f} │ (baseline)")
print(f"  ANN                   │  {mae_ann:.4f} │  {rmse_ann:.4f} │ {r2_ann:.4f} │ {ann_backend}")
print(f"  LSTM/RNN              │  {mae_lstm:.4f} │  {rmse_lstm:.4f} │   N/A   │ MAPE: {mape_lstm*100:.2f}%")
print(f"  K-Means Clustering    │   N/A   │   N/A   │  N/A   │ Silhouette: {silhouette:.4f}")
print(f"  Backpropagation       │  {mae_bp:.4f} │  {rmse_bp:.4f} │  N/A   │ Convergence: {conv_rate:.1f}%")

print(f"\n💡 KEY INSIGHTS:")
best_model = min([('Linear Regression', mae_lr), ('ANN', mae_ann), ('LSTM', mae_lstm), ('Backprop', mae_bp)], key=lambda x: x[1])
print(f"  ✓ Best regression model: {best_model[0]} (MAE = {best_model[1]:.4f})")
print(f"  ✓ Deep Learning models perform {'better' if mae_ann < mae_lr else 'worse'} than baseline")
print(f"  ✓ K-Means successfully segments regions into {3} poverty levels")
print(f"  ✓ Backprop shows {conv_rate:.1f}% loss reduction from epoch 1 to {400}")

print(f"\n🔍 RECOMMENDATIONS:")
if r2_lr > 0.7:
    print(f"  ✓ Linear model explains {r2_lr*100:.1f}% of variance - Good for interpretation")
else:
    print(f"  ⚠ Linear model explains only {r2_lr*100:.1f}% - More complex features needed")

if mae_ann < mae_lr:
    print(f"  ✓ Use ANN for production (captures non-linear patterns)")
else:
    print(f"  ✓ Linear model sufficient - simpler & faster")

print(f"  ✓ K-Means useful for regional policy targeting by poverty level")
print(f"  ✓ Ensemble predictions can improve robustness")

print("\n" + "="*80)